## 1. Environment & Path Setup
This cell dynamically checks if you are running in the cloud natively (Google Colab) or locally (Antigravity IDE), and sets the root working directory accordingly without crashing.

In [1]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Google Drive structure assumption
    PROJECT_PATH = '/content/drive/MyDrive/androidrec/Android_Recommender_System'
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"\nSuccessfully targeted project directory: {os.getcwd()}")
    else:
        print(f"\n[ERROR] Could not find {PROJECT_PATH} in Google Drive.")
        print("Please upload the Android_Recommender_System folder directly into your root MyDrive.")
else:
    print("Running locally (e.g. Antigravity IDE). Retaining current project root:")
    print(os.getcwd())

Running in Google Colab. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Successfully targeted project directory: /content/drive/MyDrive/androidrec/Android_Recommender_System


## 2. Dependency Installation & GPU Verification
Google Colab already provisions `torch`, `numpy`, and `scikit-learn`. Forcing manual re-installs of Torch pip wheels can destroy cloud GPU bindings. This cell only installs what is strictly missing (`torch-geometric`) and performs safety checks.

In [2]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("\n[CRITICAL WARNING] No GPU detected! Your graph neural network will train extremely slowly.")
    print("If on Google Colab: Go to 'Runtime' > 'Change runtime type' > Select 'T4 GPU'\n")

if IN_COLAB:
    # Quiet installation of remaining libraries required by the Colab cloud
    !pip install -q torch-geometric scikit-surprise joblib


PyTorch Version: 2.10.0+cu128
CUDA Available: True
GPU Name: Tesla T4


## 3. Data Pipeline Intercept Check
The graph requires `.npz` binary matrices to execute. If they haven't been constructed yet (e.g. fresh VM instance), this safely auto-triggers the build script.

In [3]:
!python src/build_npy.py


Loading raw myket.csv …
/content/drive/MyDrive/androidrec/Android_Recommender_System/src/build_npy.py:24: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df_raw = pd.read_csv("data/myket.csv", index_col=False, on_bad_lines="skip")
  Raw rows:  694,121
  Raw users: 10,000
  Raw apps:  7,988

After encoding:
  Interactions: 694,121
  Users: 10,000  (max=9999)
  Apps:  7,988  (max=7987)
  Saved: user2idx.pkl, app2idx.pkl, idx2app.pkl

Building app_info_sample.npy from CSV …
  Categories (30): ['Adventure', 'Artistic', 'Audio and Music', 'Book', 'Business'] …
  Apps matched & filled: 7,606 / 7,988
  Apps using zero fallback: 382
  Feature matrix shape: (7988, 33)
  Saved: data/app_info_sample.npy

Apps with features: 7,606 / 7,988
Apps using zero fallback: 382
App feature matrix shape: (7988, 33)

N = 5 held-out interactions per user per split
  Train: 594,121 interactions | 10,000 users
  Val:   50,000 interactio

## 4. Execute LightGCN on GPU
Fires the CUDA-optimized graph variant script directly locally. Metrics will output mathematically identically.

In [7]:
!python variants/variant_lightgcn_colab.py

Using device: cuda
GPU Name: Tesla T4

Loading pipeline outputs …
  Users: 10000 | Items: 7988 | Total Nodes: 17988

Building Bipartite Graph for PyG …
  Total undirected edges in graph: 948,310

Starting LightGCN Training (600 epochs) on cuda …
  Epoch 001/600 | BPR Loss: 0.6931
  Epoch 025/600 | BPR Loss: 0.2557
  Epoch 050/600 | BPR Loss: 0.1478
  Epoch 075/600 | BPR Loss: 0.1232
  Epoch 100/600 | BPR Loss: 0.1055
  Epoch 125/600 | BPR Loss: 0.0935
  Epoch 150/600 | BPR Loss: 0.0802
  Epoch 175/600 | BPR Loss: 0.0716
  Epoch 200/600 | BPR Loss: 0.0613
  Epoch 225/600 | BPR Loss: 0.0537
  Epoch 250/600 | BPR Loss: 0.0472
  Epoch 275/600 | BPR Loss: 0.0412
  Epoch 300/600 | BPR Loss: 0.0360
  Epoch 325/600 | BPR Loss: 0.0321
  Epoch 350/600 | BPR Loss: 0.0284
  Epoch 375/600 | BPR Loss: 0.0256
  Epoch 400/600 | BPR Loss: 0.0231
  Epoch 425/600 | BPR Loss: 0.0211
  Epoch 450/600 | BPR Loss: 0.0195
  Epoch 475/600 | BPR Loss: 0.0186
  Epoch 500/600 | BPR Loss: 0.0172
  Epoch 525/600 | B